# Lab 04: server, API OpenAI e parallelismo

Lab: `corso/laboratori/lab-04-01-server-e-webui, lab-04-02-client-openai, lab-04-03-parallelismo`

Esegui le celle in ordine (Runtime > Esegui tutto). Scarica fino a 1 GB e richiede alcuni minuti.


## 1. Preparazione

Scarica i binari ufficiali di llama.cpp (build `b11046`, quella del corso) e il materiale del lab.

In [ ]:
import os, subprocess
V = "b11046"
HOME = os.path.expanduser("~")
REL = f"{HOME}/llama-lab/llama-{V}"
if not os.path.exists(f"{REL}/llama"):
    subprocess.run(
        f"mkdir -p {HOME}/llama-lab && cd {HOME}/llama-lab && "
        f"curl -sLO https://github.com/ggml-org/llama.cpp/releases/download/{V}/llama-{V}-bin-ubuntu-x64.tar.gz && "
        f"tar -xzf llama-{V}-bin-ubuntu-x64.tar.gz", shell=True, check=True)
os.environ["PATH"] = REL + ":" + os.environ["PATH"]
T = min(len(os.sched_getaffinity(0)), 4)                      # thread = core disponibili (oltre i core fisici si rallenta)
os.environ["T"] = str(T)
!llama version

In [ ]:
BASE = "/content" if os.path.isdir("/content") else HOME
if not os.path.exists(f"{BASE}/corso"):
    !git clone -q --depth 1 https://github.com/JoeBosi/llama-cpp-course.git {BASE}/corso
LABS = f"{BASE}/corso/corso/laboratori"
os.environ["LABS"] = LABS

Scarica il modello di prova (785 MiB, licenza Apache 2.0) nella cache di Hugging Face.

In [ ]:
!llama download -hf ggml-org/Qwen3.5-0.8B-GGUF 2>&1 | tail -3
!llama cli -cl

## 2. Avviare il server (lab 04-01)

La funzione `avvia_server` lancia `llama serve` in background; la prossima cella sotto aspetta che sia pronto.

In [ ]:
import subprocess, time, urllib.request

def avvia_server(args, porta=8080, attesa=300):
    """Avvia `llama serve` in background e aspetta che risponda su /health."""
    log = open("/tmp/llama-server.log", "w")
    p = subprocess.Popen(["llama", "serve", *args.split(), "--host", "127.0.0.1", "--port", str(porta)],
                         stdin=subprocess.DEVNULL, stdout=log, stderr=subprocess.STDOUT)
    for _ in range(attesa):
        try:
            if urllib.request.urlopen(f"http://127.0.0.1:{porta}/health", timeout=2).status == 200:
                return p
        except Exception:
            time.sleep(1)
    raise RuntimeError("il server non risponde: " + open("/tmp/llama-server.log").read()[-1500:])

def ferma_server(p):
    p.terminate(); p.wait()

In [ ]:
p = avvia_server("-hf ggml-org/Qwen3.5-0.8B-GGUF -c 2048 -np 1 -t %d --metrics -a qwen-0.8b" % T)
print("server pronto")

In [ ]:
%%bash
curl -s http://127.0.0.1:8080/health ; echo
curl -s http://127.0.0.1:8080/v1/models | python3 -m json.tool | head -12

### Una richiesta di chat

L'API è compatibile con OpenAI: `POST /v1/chat/completions`.

In [ ]:
%%bash
curl -s http://127.0.0.1:8080/v1/chat/completions -H "Content-Type: application/json" -d '{
  "messages": [{"role": "user", "content": "Qual è la capitale d'\''Italia? Rispondi in una frase."}],
  "max_tokens": 40, "temperature": 0,
  "chat_template_kwargs": {"enable_thinking": false}}' | python3 -m json.tool | head -30

In [ ]:
%%bash
curl -s http://127.0.0.1:8080/tokenize -H "Content-Type: application/json" -d '{"content":"Ciao mondo!","with_pieces":true}'; echo
curl -s http://127.0.0.1:8080/apply-template -H "Content-Type: application/json" \
  -d '{"messages":[{"role":"system","content":"Sei un assistente."},{"role":"user","content":"Ciao"}]}'; echo

## 3. Client Python e JavaScript (lab 04-02)

In [ ]:
!pip -q install openai

In [ ]:
%%bash
cd $LABS/lab-04-02-client-openai
python3 chat.py
python3 chat.py --stream | tail -3
node --version
node chat.mjs

## 4. Parallelismo (lab 04-03)

Con `-np 1` le richieste si mettono in coda. Riavvia il server con 4 slot e confronta il throughput aggregato.

In [ ]:
%%bash
cd $LABS/lab-04-03-parallelismo
python3 carico.py --url http://127.0.0.1:8080 --richieste 4 --parallele 4 --token 24

In [ ]:
ferma_server(p)
p = avvia_server("-hf ggml-org/Qwen3.5-0.8B-GGUF -c 2048 -np 4 -t %d" % T)
print("server con 4 slot pronto")

In [ ]:
%%bash
cd $LABS/lab-04-03-parallelismo
python3 carico.py --url http://127.0.0.1:8080 --richieste 4 --parallele 4 --token 24

In [ ]:
ferma_server(p)